# VieNeu-TTS inference trên Kaggle

Notebook này chỉ là lớp hướng dẫn sử dụng repo. Toàn bộ logic nằm trong `src/inference`, nên cùng API chạy được trên Kaggle, local hoặc Modal. Kết quả mặc định là đúng một file WAV.

Trước khi chạy: bật Internet, chọn GPU, Add Input chứa model Phase 3 và file TXT. Vì repo đang private, tạo Kaggle Secret `GITHUB_TOKEN`. Tạo thêm `HF_TOKEN` có quyền đọc `neuphonic/neucodec`. Không ghi token trực tiếp vào cell.

In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
github_token = secrets.get_secret('GITHUB_TOKEN')

REPO_URL = 'https://github.com/huutamm1612/vieneu-ngoc-huyen-tts.git'
REPO_DIR = Path('/kaggle/working/vieneu-ngoc-huyen-tts')
if not REPO_DIR.exists():
    credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run(
        ['git', '-c', f'http.extraHeader=Authorization: Basic {credential}', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
        check=True,
    )
print('Repo:', REPO_DIR)
print('Python:', sys.version.split()[0])

In [ ]:
# Chỉ cần chạy một lần cho mỗi Kaggle session. Nếu Kaggle yêu cầu restart sau khi
# đổi Torch, restart session rồi tiếp tục từ cell kế tiếp.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[inference]'],
    check=True,
)
print('Đã cài inference package từ repo.')

## Cấu hình

`MODEL_PATH` phải trỏ vào thư mục full model cuối của Phase 3, không phải checkpoint Trainer tạm. Reference bên dưới giữ đúng audio và transcript từ notebook cũ. Batch logic vẫn là 128; `MAX_RUNTIME_BATCH_SIZE` chỉ chia nhỏ forward pass để vừa VRAM T4.

In [ ]:
from pathlib import Path
import torch

MODEL_PATH = '/kaggle/input/YOUR-PHASE3-MODEL/phase3_finetune/final'
INPUT_TXT = '/kaggle/input/YOUR-STORY-DATASET/story.txt'
OUTPUT_WAV = '/kaggle/working/story_complete.wav'

REF_AUDIO_PATH = '/kaggle/input/datasets/tuyetnhi811/nh-tts-dataset/ngochuyen_story_tts_clean/raw_audio/ngochuyen_00769.wav'
REF_TEXT = 'Vì thế, Đồng chí luôn được các Đồng chí lãnh đạo cấp cao của Đảng tin tưởng, đánh giá cao.'

TEST_MIN_CHARS = 80
TEST_TARGET_CHARS = 128
TEST_MAX_CHARS = 156
TEST_BATCH_SIZE = 128
TEST_MAX_LENGTH_GAP = 12

GPU_COUNT = torch.cuda.device_count()
NUM_GPUS = min(2, GPU_COUNT) if GPU_COUNT else 1
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
MAX_RUNTIME_BATCH_SIZE = 8 if any('T4' in name for name in GPU_NAMES) else None

assert Path(INPUT_TXT).is_file(), f'Không tìm thấy TXT: {INPUT_TXT}'
assert Path(REF_AUDIO_PATH).is_file(), f'Không tìm thấy reference: {REF_AUDIO_PATH}'
if MODEL_PATH.startswith('/'):
    assert Path(MODEL_PATH).is_dir(), f'Không tìm thấy model: {MODEL_PATH}'
print('GPU:', GPU_NAMES or ['CPU'])
print('Logical batch:', TEST_BATCH_SIZE, '| runtime batch:', MAX_RUNTIME_BATCH_SIZE or 'auto')

In [ ]:
from inference import InferenceConfig, TTSInference, prepare_batches

batches = prepare_batches(
    input_path=INPUT_TXT,
    min_chars=TEST_MIN_CHARS,
    target_chars=TEST_TARGET_CHARS,
    max_chars=TEST_MAX_CHARS,
    batch_size=TEST_BATCH_SIZE,
    max_length_gap=TEST_MAX_LENGTH_GAP,
)
items = [item for batch in batches for item in batch]
print(f'{len(items)} đoạn trong {len(batches)} logical batches')
print('Đoạn đầu:', min(items, key=lambda item: item['index'])['text'])

In [ ]:
config = InferenceConfig(
    model=MODEL_PATH,
    devices='auto',
    num_gpus=NUM_GPUS,
    min_chars=TEST_MIN_CHARS,
    target_chars=TEST_TARGET_CHARS,
    max_chars=TEST_MAX_CHARS,
    batch_size=TEST_BATCH_SIZE,
    max_length_gap=TEST_MAX_LENGTH_GAP,
    max_runtime_batch_size=MAX_RUNTIME_BATCH_SIZE,
    do_sample=False,
    keep_segments=False,
)

with TTSInference(config) as tts:
    result = tts.infer(
        batches=batches,
        reference_audio=REF_AUDIO_PATH,
        reference_text=REF_TEXT,
        output_path=OUTPUT_WAV,
    )

print(result.as_dict())

In [ ]:
from IPython.display import Audio, FileLink, display

display(Audio(OUTPUT_WAV))
display(FileLink(OUTPUT_WAV))

## Cách gọi ngắn hơn

Nếu không cần xem/sửa batches, bỏ cell `prepare_batches` và truyền thẳng `input_path=INPUT_TXT` vào `tts.infer(...)`; vì `batches=None`, pipeline sẽ tự tiền xử lý với các giá trị trong `InferenceConfig`.